# Swiss Citation Retrieval: Segment-Lattice Funnel V3

Pipeline:

1. Build a fresh segment/option registry from `data_insights/*_classified_citations.jsonl` (re-parsed with the current parser).
2. The build step also computes **court co-signals**: for every BGE division and docket prefix, the top law codes co-cited in the court consideration text and the top distinctive German keywords.
3. Inspect cards to confirm court segments+options carry meaning, examples, and co-signals.
4. Run an **oracle** plan from gold to measure the funnel ceiling (law + court).
5. Run the **LLM planner** with the enriched cards. Its `segment_decisions` JSONL is fed back into the same `run-candidates` funnel.
6. Audit and trace per-stage law/court recall + final candidate count for both runs.

In [ ]:
import os
import sys
import json
import subprocess
from pathlib import Path

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

BASE_DIR = Path("/content/drive/MyDrive/swiss_law") if IN_COLAB else Path("..").resolve()
DATA_DIR = BASE_DIR / "data"
INSIGHTS_DIR = BASE_DIR / "data_insights"
ART_DIR = BASE_DIR / "artifacts"
SCRIPT_DIR = BASE_DIR / "scripts"

sys.path.insert(0, str(SCRIPT_DIR))
ART_DIR.mkdir(parents=True, exist_ok=True)

print("BASE_DIR    :", BASE_DIR)
print("DATA_DIR    :", DATA_DIR)
print("INSIGHTS_DIR:", INSIGHTS_DIR)
print("ART_DIR     :", ART_DIR)
print("SCRIPT_DIR  :", SCRIPT_DIR)
assert (SCRIPT_DIR / "segment_lattice_v3.py").exists(), "Copy scripts/segment_lattice_v3.py into BASE_DIR/scripts first"
assert (SCRIPT_DIR / "extract_citation_graph.py").exists(), "Copy scripts/extract_citation_graph.py into BASE_DIR/scripts first"

Mounted at /content/drive
BASE_DIR    : /content/drive/MyDrive/swiss_law
DATA_DIR    : /content/drive/MyDrive/swiss_law/data
INSIGHTS_DIR: /content/drive/MyDrive/swiss_law/data_insights
ART_DIR     : /content/drive/MyDrive/swiss_law/artifacts
SCRIPT_DIR  : /content/drive/MyDrive/swiss_law/scripts


## 1. Optional Fresh Extraction

Run only when parser logic changed. Recreates classified citation JSONL from source CSVs.

In [ ]:
REFRESH_EXTRACTION = False

if REFRESH_EXTRACTION:
    subprocess.check_call([
        sys.executable,
        str(SCRIPT_DIR / "extract_citation_graph.py"),
        "--data-dir", str(DATA_DIR),
        "--output-dir", str(INSIGHTS_DIR),
        "--db-name", "citation_graph_extracted.sqlite",
    ])
else:
    print("Skipping extraction refresh. Set REFRESH_EXTRACTION=True when parser/data changed.")

Skipping extraction refresh. Set REFRESH_EXTRACTION=True when parser/data changed.


## 2. Invalidate Old Funnel Outputs

Removes only stale coarse-funnel artifacts (`recall_funnel.duckdb`, `choice_cards.json`, old candidate/planner JSONLs). Source data and `data_insights` are untouched.

In [ ]:
from segment_lattice_v3 import invalidate_stale_artifacts
import segment_lattice_v3, inspect
print("court_funnel" in inspect.getsource(segment_lattice_v3.hard_filter_base))
# True  -> new code is loaded
# False -> still the old script in Drive

INVALIDATE_OLD_FUNNEL_ARTIFACTS = True
if INVALIDATE_OLD_FUNNEL_ARTIFACTS:
    removed = invalidate_stale_artifacts(ART_DIR)
    print("Removed", len(removed), "old artifacts")
    for path in removed:
        print(path)
else:
    print("Old artifacts left untouched. Set INVALIDATE_OLD_FUNNEL_ARTIFACTS=True to remove them.")

True
Removed 0 old artifacts


## 3. Build Segment-Lattice Index, Cards, And Court Co-Signals

Schema bumped to `..._b_cosignals` so this rebuild also computes:

- per-option `co_signals`: top law codes co-cited in court consideration text per `division`/`docket_prefix`
- per-option `text_keywords`: top distinctive German nouns from the same text

These give the LLM planner the law→court signal as **corpus evidence**, not a hardcoded mapping. Court CSV scan adds ~3–8 minutes the first time.

In [ ]:
from segment_lattice_v3 import build_index

DB_PATH = ART_DIR / "segment_lattice_v3.sqlite"
CARDS_PATH = ART_DIR / "choice_cards_v3.json"

build_index(
    data_dir=DATA_DIR,
    insights_dir=INSIGHTS_DIR,
    db_path=DB_PATH,
    cards_path=CARDS_PATH,
    force=True,
)
print("DB_PATH   :", DB_PATH)
print("CARDS_PATH:", CARDS_PATH)

laws_de: done scanned=197,945 source_rows=175,933
court_considerations: scanned=500,000 source_rows=450,934
court_considerations: scanned=1,000,000 source_rows=907,418
court_considerations: scanned=1,500,000 source_rows=1,365,237
court_considerations: scanned=2,000,000 source_rows=1,772,053
court_considerations: done scanned=2,416,056 source_rows=1,985,178
Computing court co-signals from court_considerations.csv text...
  co-signals scan: 250,000 rows, 250,000 matched
  co-signals scan: 500,000 rows, 500,000 matched
  co-signals scan: 750,000 rows, 750,000 matched
  co-signals scan: 1,000,000 rows, 1,000,000 matched
  co-signals scan: 1,250,000 rows, 1,250,000 matched
  co-signals scan: 1,500,000 rows, 1,500,000 matched
  co-signals scan: 1,750,000 rows, 1,750,000 matched
  co-signals scan: 2,000,000 rows, 2,000,000 matched
  co-signals scan: 2,250,000 rows, 2,250,000 matched
co-signals built: court rows scanned=2,476,315, matched_to_segments=2,476,315, options_with_signals=69
Built v3

## 4. Verify Court Cards Carry Co-Signals

Hard check that the LLM will receive court segments+options correctly. We assert:

- court has at least the BGE divisions and the major docket prefixes
- semantic court options have non-empty `co_signals` for the most common buckets

In [ ]:
cards = json.loads(CARDS_PATH.read_text(encoding="utf-8"))
print("card groups:", len(cards))

court_groups = sorted(k for k in cards if k.startswith("court_considerations."))
print("court groups:", court_groups)

def show_card(card_key, n=5):
    if card_key not in cards:
        print("missing:", card_key); return
    print(f"\n=== {card_key} ({len(cards[card_key])} options) ===")
    for opt in cards[card_key][:n]:
        co = opt.get("co_signals") or []
        kw = opt.get("text_keywords") or []
        co_str = ", ".join(f"{c['law_code']}({c['freq']})" for c in co[:5]) or "-"
        kw_str = ", ".join(kw[:6]) or "-"
        print(f"  id={opt['id']:<8} src={opt['source_count']:>7} kind={opt['selector_kind']:<12} co=[{co_str}]")
        print(f"           keywords=[{kw_str}]")

show_card("court_considerations.court.court_bge.division")
show_card("court_considerations.court.court_case.docket_prefix", n=10)

# Sanity assertions
div_card = cards.get("court_considerations.court.court_bge.division", [])
div_ids = {o["id"] for o in div_card}
assert div_ids >= {"I", "II", "III", "IV", "V"}, f"BGE divisions missing: have {div_ids}"
assert any(o.get("co_signals") for o in div_card), "BGE division cards have no co_signals \u2014 rebuild needed"

prefix_card = cards.get("court_considerations.court.court_case.docket_prefix", [])
prefix_ids = {o["id"] for o in prefix_card}
for must in {"1B", "4A", "5A", "6B", "8C", "9C", "2C", "7B"}:
    assert must in prefix_ids, f"docket_prefix {must} missing from registry"
with_co = sum(1 for o in prefix_card if o.get("co_signals"))
print(f"\ndocket_prefix options with co_signals: {with_co} / {len(prefix_card)}")
assert with_co >= 10, "Too few docket_prefix options carry co_signals; the LLM cannot infer law\u2192court from cards."

card groups: 48
court groups: ['court_considerations.court.court_bge.court_base', 'court_considerations.court.court_bge.division', 'court_considerations.court.court_bge.page', 'court_considerations.court.court_bge.pattern', 'court_considerations.court.court_bge.pinpoint', 'court_considerations.court.court_bge.pinpoint_unit', 'court_considerations.court.court_bge.reporter', 'court_considerations.court.court_bge.subfamily', 'court_considerations.court.court_bge.volume', 'court_considerations.court.court_case.consideration', 'court_considerations.court.court_case.court_base', 'court_considerations.court.court_case.court_chamber', 'court_considerations.court.court_case.decision_date', 'court_considerations.court.court_case.decision_year', 'court_considerations.court.court_case.docket', 'court_considerations.court.court_case.docket_prefix', 'court_considerations.court.court_case.legal_area_code', 'court_considerations.court.court_case.pattern', 'court_considerations.court.court_case.separat

## 5. Build A Compact LLM-Ready Card Bundle

The full registry has ~175 docket prefixes and many subfamily/separator options. The LLM planner does not need the long tail. We trim per segment to the top semantic options + structural anchors and persist a compact JSON for prompts.

In [ ]:
COMPACT_CARDS_PATH = ART_DIR / "choice_cards_v3_compact.json"

TOP_PER_SEGMENT = {
    "laws_de.law.statute_article.law_code": 80,
    "court_considerations.court.court_bge.division": 5,
    "court_considerations.court.court_case.docket_prefix": 30,
    "court_considerations.court.court_case.legal_area_code": 30,
}

def shrink_option(o):
    return {
        "id": o["id"],
        "meaning_en": o["meaning_en"],
        "source_count": o["source_count"],
        "selector_kind": o["selector_kind"],
        "examples": (o.get("examples") or [])[:3],
        "co_signals": [{"law_code": c["law_code"], "freq": c["freq"]} for c in (o.get("co_signals") or [])[:6]],
        "text_keywords": (o.get("text_keywords") or [])[:6],
    }

compact = {}
for key, options in cards.items():
    options_sorted = sorted(options, key=lambda o: -o.get("source_count", 0))
    if key in TOP_PER_SEGMENT:
        compact[key] = [shrink_option(o) for o in options_sorted[: TOP_PER_SEGMENT[key]]]
    elif any(o["selector_kind"] == "semantic" for o in options):
        compact[key] = [shrink_option(o) for o in options_sorted if o["selector_kind"] == "semantic"][:50]

COMPACT_CARDS_PATH.write_text(json.dumps(compact, ensure_ascii=False, indent=2), encoding="utf-8")
print("compact card groups:", list(compact.keys()))
print("compact size (bytes):", COMPACT_CARDS_PATH.stat().st_size)

compact card groups: ['court_considerations.court.court_bge.division', 'court_considerations.court.court_case.docket_prefix', 'court_considerations.court.court_case.legal_area_code', 'court_considerations.unknown.unknown.docket_prefix', 'court_considerations.unknown.unknown.legal_area_code', 'laws_de.law.statute_article.law_code']
compact size (bytes): 130568


## 6. Oracle Ceiling — What Recall Is Possible?

The oracle reads gold, looks up each gold citation’s `law_code` / `division` / `docket_prefix` from the registry, and emits the perfect `segment_decisions` for the funnel. We then run the **same** `run-candidates` and `audit` steps the LLM uses.

Interpretation:
- oracle law_recall ≈ 1.0 and oracle court_recall ≈ 1.0 → funnel/registry are sound.
- gap < 1.0 → a parser/registry bug (reported in dropped-gold CSV) is the bottleneck.

In [ ]:
from segment_lattice_v3 import oracle_plan_for_split, run_candidates, audit_candidates

RUN_SPLIT = "val"  # "val", "test", or "train"
LAW_BUDGET = 2_000
COURT_BUDGET = 2_000

ORACLE_PLAN = ART_DIR / f"{RUN_SPLIT}_segment_lattice_v3_oracle_planner_outputs.jsonl"
ORACLE_CAND = ART_DIR / f"{RUN_SPLIT}_segment_lattice_v3_candidate_sets.jsonl"
ORACLE_SUMMARY = ART_DIR / f"{RUN_SPLIT}_segment_lattice_v3_oracle_candidate_summary.csv"
ORACLE_DROPPED = ART_DIR / f"{RUN_SPLIT}_segment_lattice_v3_oracle_dropped_gold.csv"

oracle_plan_for_split(split=RUN_SPLIT, data_dir=DATA_DIR, db_path=DB_PATH, out_dir=ART_DIR)

run_candidates(
    split=RUN_SPLIT,
    data_dir=DATA_DIR,
    db_path=DB_PATH,
    out_dir=ART_DIR,
    law_budget=LAW_BUDGET,
    court_budget=COURT_BUDGET,
    planner_input=ORACLE_PLAN,
)

if RUN_SPLIT in {"train", "val"}:
    audit_candidates(split=RUN_SPLIT, data_dir=DATA_DIR, db_path=DB_PATH, out_dir=ART_DIR)
    # Snapshot the oracle outputs so the LLM run does not overwrite them.
    import shutil
    shutil.copyfile(ART_DIR / f"{RUN_SPLIT}_segment_lattice_v3_candidate_summary.csv", ORACLE_SUMMARY)
    shutil.copyfile(ART_DIR / f"{RUN_SPLIT}_segment_lattice_v3_dropped_gold.csv", ORACLE_DROPPED)
    shutil.copyfile(ORACLE_CAND, ART_DIR / f"{RUN_SPLIT}_segment_lattice_v3_oracle_candidate_sets.jsonl")
    print("\nOracle outputs snapshotted:")
    print(" ", ORACLE_SUMMARY)
    print(" ", ORACLE_DROPPED)

Saved oracle planner output: /content/drive/MyDrive/swiss_law/artifacts/val_segment_lattice_v3_oracle_planner_outputs.jsonl
Next: python scripts/segment_lattice_v3.py run-candidates --split val --planner-input /content/drive/MyDrive/swiss_law/artifacts/val_segment_lattice_v3_oracle_planner_outputs.jsonl
Then : python scripts/segment_lattice_v3.py audit --split val
val_001 law= 2000 court= 2000 total= 4000
val_002 law= 954 court= 2000 total= 2954
val_003 law= 2000 court= 2000 total= 4000
val_004 law= 2000 court= 2000 total= 4000
val_005 law= 2000 court= 2000 total= 4000
val_006 law= 2000 court= 2000 total= 4000
val_007 law= 2000 court= 2000 total= 4000
val_008 law= 2000 court= 2000 total= 4000
val_009 law= 2000 court= 2000 total= 4000
val_010 law= 2000 court= 2000 total= 4000
Saved candidates: /content/drive/MyDrive/swiss_law/artifacts/val_segment_lattice_v3_candidate_sets.jsonl
Saved plans     : /content/drive/MyDrive/swiss_law/artifacts/val_segment_lattice_v3_planner_outputs.jsonl
Sav

## 7. LLM Planner Cell (Qwen3-32B)

Loads `Qwen/Qwen3-32B` via Hugging Face Transformers (Colab GPU) and runs a two-pass prompt per query:

- **Pass 1** picks `law_code` ids from the compact card list.
- **Pass 2** picks `division` and `docket_prefix` ids using the law codes from pass 1 and the **co_signals** + **text_keywords** in the court cards.

Each output is parsed into the funnel's `segment_decisions` schema and validated against the registry via `validate_external_decisions`. Any unknown id the model emits is silently dropped — the funnel never receives invalid options.

Set `USE_HEURISTIC_FALLBACK = True` to skip the LLM and use the data-driven heuristic instead (useful for a quick smoke test before paying for GPU).

In [ ]:
USE_HEURISTIC_FALLBACK = False  # set True to bypass Qwen and use the registry-only heuristic.
USE_HYBRID_PLANNER = True  # union LLM + heuristic + parser decisions, so LLM omissions never kill court recall.
LOG_FIRST_N_QWEN_RAW = 3  # print raw Qwen responses for the first N queries to diagnose JSON / empty-array issues.

QWEN_MODEL_ID = "Qwen/Qwen3-32B"
QWEN_MAX_NEW_TOKENS = 512
QWEN_TEMPERATURE = 0.0  # deterministic option selection
QWEN_LOAD_IN_4BIT = False  # 95 GB VRAM (G4): full bf16 fits (~64 GB weights + activations)
QWEN_USE_FLASH_ATTN_2 = True  # set False if flash-attn is not installed in the env

LLM_PLAN_PATH = ART_DIR / f"{RUN_SPLIT}_segment_lattice_v3_llm_planner_outputs.jsonl"

import csv
import re
import sqlite3
from segment_lattice_v3 import derive_decisions, validate_external_decisions

compact = json.loads(COMPACT_CARDS_PATH.read_text(encoding="utf-8"))
law_card = compact.get("laws_de.law.statute_article.law_code", [])
div_card = compact.get("court_considerations.court.court_bge.division", [])
prefix_card = compact.get("court_considerations.court.court_case.docket_prefix", [])
LAW_IDS = {o["id"] for o in law_card}
DIV_IDS = {o["id"] for o in div_card}
PREFIX_IDS = {o["id"] for o in prefix_card}

def render_law_pass_prompt(query):
    return (
        "You are choosing Swiss law codes that the query is about.\n"
        "Return strict JSON only: {\"law_codes\": [<id>, ...]}\n"
        "Use only ids from the candidate list. Empty list if none apply.\n\n"
        f"Query: {query}\n\n"
        "Candidates (id - meaning):\n" +
        "\n".join(f"  {o['id']}: {o['meaning_en']}" for o in law_card)
    )

def render_court_pass_prompt(query, picked_law_codes):
    def fmt(o):
        co = ", ".join(f"{c['law_code']}({c['freq']})" for c in o.get("co_signals", []))
        kw = ", ".join(o.get("text_keywords", []))
        return f"  {o['id']}: {o['meaning_en']} | co_signals=[{co}] | keywords=[{kw}]"
    return (
        "You are choosing Federal Tribunal court divisions and docket prefixes that the query is about.\n"
        "Use co_signals (law codes co-cited in court text) and keywords as evidence.\n"
        "IMPORTANT: most Swiss legal queries DO cite court decisions. Pick the divisions and prefixes whose co_signals best match the law codes already chosen, even if you are not sure. Returning empty lists is rarely correct.\n"
        "Return strict JSON only: {\"divisions\": [...], \"divisions_maybe\": [...], \"docket_prefixes\": [...], \"docket_prefixes_maybe\": [...]}\n"
        "Use 'maybe' for plausible-but-unsure picks. Use only ids from the candidate lists. Empty lists are allowed but discouraged.\n\n"
        f"Query: {query}\n"
        f"Law codes already chosen: {picked_law_codes}\n\n"
        "BGE divisions:\n" + "\n".join(fmt(o) for o in div_card) +
        "\n\nDocket prefixes:\n" + "\n".join(fmt(o) for o in prefix_card)
    )

# ---------- Qwen3-32B loader ----------
_qwen_state = {"tokenizer": None, "model": None}

def _load_qwen():
    if _qwen_state["model"] is not None:
        return _qwen_state["tokenizer"], _qwen_state["model"]
    print(f"Loading {QWEN_MODEL_ID} in {'4-bit' if QWEN_LOAD_IN_4BIT else 'bf16'}... (first call only)")
    import torch
    from transformers import AutoModelForCausalLM, AutoTokenizer
    tok = AutoTokenizer.from_pretrained(QWEN_MODEL_ID, trust_remote_code=True)
    kwargs = {"trust_remote_code": True, "device_map": "auto", "torch_dtype": torch.bfloat16}
    if QWEN_LOAD_IN_4BIT:
        from transformers import BitsAndBytesConfig
        kwargs["quantization_config"] = BitsAndBytesConfig(
            load_in_4bit=True,
            bnb_4bit_compute_dtype=torch.bfloat16,
            bnb_4bit_quant_type="nf4",
        )
    if QWEN_USE_FLASH_ATTN_2:
        try:
            import flash_attn  # noqa: F401
            kwargs["attn_implementation"] = "flash_attention_2"
        except Exception:
            print("  flash-attn not installed; using default attention.")
    model = AutoModelForCausalLM.from_pretrained(QWEN_MODEL_ID, **kwargs)
    model.eval()
    _qwen_state["tokenizer"] = tok
    _qwen_state["model"] = model
    print(f"  loaded. device map: {getattr(model, 'hf_device_map', 'single')}")
    if torch.cuda.is_available():
        used_gb = torch.cuda.memory_allocated() / (1024 ** 3)
        print(f"  GPU mem allocated after load: {used_gb:.1f} GB")
    return tok, model

def qwen_chat(prompt, max_new_tokens=QWEN_MAX_NEW_TOKENS, temperature=QWEN_TEMPERATURE):
    import torch
    tok, model = _load_qwen()
    messages = [
        {"role": "system", "content": "You are a precise option selector. Always reply with strict JSON only, no prose, no markdown fences."},
        {"role": "user", "content": prompt},
    ]
    text = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)
    inputs = tok([text], return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=temperature > 0.0,
            temperature=max(temperature, 1e-5),
            top_p=0.9 if temperature > 0.0 else 1.0,
            pad_token_id=tok.eos_token_id,
        )
    gen = out[0][inputs.input_ids.shape[1]:]
    return tok.decode(gen, skip_special_tokens=True).strip()

_JSON_RE = re.compile(r"\{.*\}", re.DOTALL)

def _parse_json_block(text):
    m = _JSON_RE.search(text or "")
    if not m:
        return {}
    try:
        return json.loads(m.group(0))
    except Exception:
        for end in range(len(m.group(0)), 0, -1):
            try:
                return json.loads(m.group(0)[:end])
            except Exception:
                continue
    return {}

def _filter_ids(values, allowed):
    return sorted({str(v) for v in (values or []) if str(v) in allowed})

def qwen_planner(query, conn, log_raw=False):
    """Returns (decisions, debug_info). debug_info is a dict with raw responses."""
    debug = {}
    law_raw = qwen_chat(render_law_pass_prompt(query))
    debug["law_raw"] = law_raw
    if log_raw:
        print("  --- qwen LAW raw ---\n", law_raw[:600], "\n  --------------------")
    law_obj = _parse_json_block(law_raw)
    picked_law = _filter_ids(law_obj.get("law_codes"), LAW_IDS)

    parser_law, _ = derive_decisions(conn, query)
    for d in parser_law:
        if d["segment_key"] == "law_code":
            for v in d["selected_option_ids"]:
                if v in LAW_IDS and v not in picked_law:
                    picked_law.append(v)
    picked_law = sorted(set(picked_law))

    court_raw = qwen_chat(render_court_pass_prompt(query, picked_law))
    debug["court_raw"] = court_raw
    if log_raw:
        print("  --- qwen COURT raw ---\n", court_raw[:600], "\n  ----------------------")
    court_obj = _parse_json_block(court_raw)
    div_inc = _filter_ids(court_obj.get("divisions"), DIV_IDS)
    div_maybe = [v for v in _filter_ids(court_obj.get("divisions_maybe"), DIV_IDS) if v not in div_inc]
    pref_inc = _filter_ids(court_obj.get("docket_prefixes"), PREFIX_IDS)
    pref_maybe = [v for v in _filter_ids(court_obj.get("docket_prefixes_maybe"), PREFIX_IDS) if v not in pref_inc]

    decisions = []
    if picked_law:
        decisions.append({
            "funnel": "law", "dataset": "laws_de", "family": "law",
            "segment_key": "law_code", "selected_option_ids": picked_law,
            "mode": "include", "confidence": 0.95, "reason": "qwen3-32b law pass",
        })
    if div_inc:
        decisions.append({"funnel": "court", "dataset": "court_considerations", "family": "court_funnel",
                          "segment_key": "division", "selected_option_ids": div_inc,
                          "mode": "include", "confidence": 0.9, "reason": "qwen3-32b court pass (include)"})
    if div_maybe:
        decisions.append({"funnel": "court", "dataset": "court_considerations", "family": "court_funnel",
                          "segment_key": "division", "selected_option_ids": div_maybe,
                          "mode": "maybe", "confidence": 0.5, "reason": "qwen3-32b court pass (maybe)"})
    if pref_inc:
        decisions.append({"funnel": "court", "dataset": "court_considerations", "family": "court_funnel",
                          "segment_key": "docket_prefix", "selected_option_ids": pref_inc,
                          "mode": "include", "confidence": 0.9, "reason": "qwen3-32b court pass (include)"})
    if pref_maybe:
        decisions.append({"funnel": "court", "dataset": "court_considerations", "family": "court_funnel",
                          "segment_key": "docket_prefix", "selected_option_ids": pref_maybe,
                          "mode": "maybe", "confidence": 0.5, "reason": "qwen3-32b court pass (maybe)"})
    return validate_external_decisions(conn, decisions), debug

def heuristic_planner(query, conn):
    """Registry-only co_signals heuristic. Always emits BGE divisions and at least
    the top docket prefixes whose co_signals overlap the picked law codes; if no
    law code is detected, emits all 5 divisions + top 8 prefixes as `mode=maybe`
    so the funnel still has court coverage."""
    law_decisions, _ = derive_decisions(conn, query)
    picked_law = sorted({v for d in law_decisions if d["segment_key"] == "law_code" for v in d["selected_option_ids"]})
    decisions = list(law_decisions)
    if picked_law:
        picked_law_set = set(picked_law)
        div_pick = [o["id"] for o in div_card if any(c["law_code"] in picked_law_set for c in o.get("co_signals", []))]
        prefix_pick = [o["id"] for o in prefix_card if any(c["law_code"] in picked_law_set for c in o.get("co_signals", []))]
    else:
        # No law signal -> open court coverage with all divisions and the most
        # populous prefixes (still bounded; this is mode=maybe, not include).
        div_pick = [o["id"] for o in div_card]
        prefix_pick = [o["id"] for o in prefix_card[:12]]
    if div_pick:
        decisions.append({
            "funnel": "court", "dataset": "court_considerations", "family": "court_funnel",
            "segment_key": "division", "selected_option_ids": div_pick, "mode": "maybe",
            "confidence": 0.6, "reason": "heuristic: co_signals (or fallback) on division",
        })
    if prefix_pick:
        decisions.append({
            "funnel": "court", "dataset": "court_considerations", "family": "court_funnel",
            "segment_key": "docket_prefix", "selected_option_ids": prefix_pick, "mode": "maybe",
            "confidence": 0.6, "reason": "heuristic: co_signals (or fallback) on docket_prefix",
        })
    return validate_external_decisions(conn, decisions)

def merge_two_plans(a, b):
    """Union two decision lists by (dataset, family, segment_key, mode)."""
    out_map = {}
    for d in (a or []) + (b or []):
        key = (d["dataset"], d["family"], d["segment_key"], d["mode"])
        if key not in out_map:
            out_map[key] = dict(d)
            out_map[key]["selected_option_ids"] = list(d["selected_option_ids"])
        else:
            existing = set(out_map[key]["selected_option_ids"])
            for v in d["selected_option_ids"]:
                if v not in existing:
                    out_map[key]["selected_option_ids"].append(v)
                    existing.add(v)
    for d in out_map.values():
        d["selected_option_ids"] = sorted(set(d["selected_option_ids"]))
    return list(out_map.values())

def hybrid_planner(query, conn, log_raw=False):
    """LLM include picks + heuristic maybe picks + parser explicit decisions, unioned.
    Hard-filter is driven by include-mode picks; heuristic adds maybe-mode coverage so
    omissions in the LLM cannot zero out the court funnel."""
    llm_decisions, debug = qwen_planner(query, conn, log_raw=log_raw)
    heur_decisions = heuristic_planner(query, conn)
    merged = merge_two_plans(llm_decisions, heur_decisions)
    return validate_external_decisions(conn, merged), debug

# ---------- Run planner over the split ----------
with open(DATA_DIR / f"{RUN_SPLIT}.csv", encoding="utf-8-sig") as f:
    rows = [{"query_id": r["query_id"], "query": r["query"]} for r in csv.DictReader(f)]

conn = sqlite3.connect(DB_PATH)
debug_log = []
with open(LLM_PLAN_PATH, "w", encoding="utf-8", newline="\n") as out:
    for i, row in enumerate(rows, 1):
        log_raw = (i <= LOG_FIRST_N_QWEN_RAW) and not USE_HEURISTIC_FALLBACK
        if USE_HEURISTIC_FALLBACK:
            decisions = heuristic_planner(row["query"], conn)
            debug = {}
        else:
            try:
                if USE_HYBRID_PLANNER:
                    decisions, debug = hybrid_planner(row["query"], conn, log_raw=log_raw)
                else:
                    decisions, debug = qwen_planner(row["query"], conn, log_raw=log_raw)
            except Exception as e:
                print(f"  [{row['query_id']}] qwen failure ({e}); falling back to heuristic")
                decisions = heuristic_planner(row["query"], conn)
                debug = {"error": str(e)}
        debug_log.append({"query_id": row["query_id"], **debug})
        out.write(json.dumps({"query_id": row["query_id"], "segment_decisions": decisions}, ensure_ascii=False) + "\n")
        if i % 10 == 0 or i == len(rows):
            print(f"  planned {i}/{len(rows)}", flush=True)
conn.close()

# Persist Qwen raw responses for debugging
DEBUG_PATH = ART_DIR / f"{RUN_SPLIT}_segment_lattice_v3_llm_raw_log.jsonl"
with open(DEBUG_PATH, "w", encoding="utf-8") as f:
    for d in debug_log:
        f.write(json.dumps(d, ensure_ascii=False) + "\n")
print("Wrote LLM planner outputs:", LLM_PLAN_PATH)
print("Wrote LLM raw log       :", DEBUG_PATH)


Loading Qwen/Qwen3-32B in bf16... (first call only)


config.json:   0%|          | 0.00/728 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

Fetching 17 files:   0%|          | 0/17 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/707 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


  loaded. device map: single
  GPU mem allocated after load: 61.0 GB
  --- qwen LAW raw ---
 ```json
{"law_codes": ["StPO", "BV"]}
``` 
  --------------------
  --- qwen COURT raw ---
 ```json
{
  "divisions": ["IV"],
  "divisions_maybe": ["I", "II"],
  "docket_prefixes": ["1B", "6P", "7B"],
  "docket_prefixes_maybe": ["2P", "4D", "5P"]
}
``` 
  ----------------------
  --- qwen LAW raw ---
 ```json
{"law_codes": ["IVG", "IVV"]}
``` 
  --------------------
  --- qwen COURT raw ---
 ```json
{
  "divisions": ["V"],
  "divisions_maybe": ["II", "I"],
  "docket_prefixes": ["8C", "9C", "I"],
  "docket_prefixes_maybe": ["C", "B", "H"]
}
``` 
  ----------------------
  --- qwen LAW raw ---
 ```json
{"law_codes": ["StPO"]}
``` 
  --------------------
  --- qwen COURT raw ---
 ```json
{
  "divisions": ["IV"],
  "divisions_maybe": ["I", "II"],
  "docket_prefixes": ["6B", "1B", "7B", "6P"],
  "docket_prefixes_maybe": ["4A", "5P", "2P"]
}
``` 
  ----------------------
  planned 10/10
Wrote LLM plan

## 8. Run Candidates Through The Funnel With The LLM Plan And Audit

Same `run-candidates` and `audit` functions used by oracle. Reads no gold.

In [ ]:
LLM_SUMMARY = ART_DIR / f"{RUN_SPLIT}_segment_lattice_v3_llm_candidate_summary.csv"
LLM_DROPPED = ART_DIR / f"{RUN_SPLIT}_segment_lattice_v3_llm_dropped_gold.csv"
LLM_CAND = ART_DIR / f"{RUN_SPLIT}_segment_lattice_v3_llm_candidate_sets.jsonl"

run_candidates(
    split=RUN_SPLIT,
    data_dir=DATA_DIR,
    db_path=DB_PATH,
    out_dir=ART_DIR,
    law_budget=LAW_BUDGET,
    court_budget=COURT_BUDGET,
    planner_input=LLM_PLAN_PATH,
)
if RUN_SPLIT in {"train", "val"}:
    audit_candidates(split=RUN_SPLIT, data_dir=DATA_DIR, db_path=DB_PATH, out_dir=ART_DIR)
    import shutil
    shutil.copyfile(ART_DIR / f"{RUN_SPLIT}_segment_lattice_v3_candidate_summary.csv", LLM_SUMMARY)
    shutil.copyfile(ART_DIR / f"{RUN_SPLIT}_segment_lattice_v3_dropped_gold.csv", LLM_DROPPED)
    shutil.copyfile(ART_DIR / f"{RUN_SPLIT}_segment_lattice_v3_candidate_sets.jsonl", LLM_CAND)
    print("\nLLM outputs snapshotted:")
    print(" ", LLM_SUMMARY)
    print(" ", LLM_DROPPED)

val_001 law= 1964 court= 2000 total= 3964
val_002 law= 870 court= 2000 total= 2870
val_003 law= 1306 court= 2000 total= 3306
val_004 law= 2000 court= 2000 total= 4000
val_005 law= 2000 court= 2000 total= 4000
val_006 law= 2000 court= 2000 total= 4000
val_007 law= 2000 court= 2000 total= 4000
val_008 law= 2000 court= 2000 total= 4000
val_009 law= 2000 court= 2000 total= 4000
val_010 law= 2000 court= 2000 total= 4000
Saved candidates: /content/drive/MyDrive/swiss_law/artifacts/val_segment_lattice_v3_candidate_sets.jsonl
Saved plans     : /content/drive/MyDrive/swiss_law/artifacts/val_segment_lattice_v3_planner_outputs.jsonl
Saved summary: /content/drive/MyDrive/swiss_law/artifacts/val_segment_lattice_v3_candidate_summary.csv
Saved dropped: /content/drive/MyDrive/swiss_law/artifacts/val_segment_lattice_v3_dropped_gold.csv

LLM outputs snapshotted:
  /content/drive/MyDrive/swiss_law/artifacts/val_segment_lattice_v3_llm_candidate_summary.csv
  /content/drive/MyDrive/swiss_law/artifacts/val_

## 9. Per-Stage Recall Trace (Law + Court Together)

For every candidate JSONL, replay each stage in `stage_diagnostics` against gold and report:

- **stage**: `source` → `hard_filter:*` → `topk_budget`
- **law_recall** and **court_recall** after that stage
- **candidate_count** at that stage

We can pinpoint exactly which segment decision dropped recall.

In [ ]:
import sqlite3
import statistics

conn = sqlite3.connect(DB_PATH)
family_by_citation = {row[1]: ("law" if row[0] == "laws_de" else "court") for row in conn.execute("SELECT dataset, citation FROM source_citations")}

def load_gold(split):
    out = {}
    with open(DATA_DIR / f"{split}.csv", encoding="utf-8-sig") as f:
        for row in csv.DictReader(f):
            cites = [c.strip() for c in (row.get("gold_citations") or "").split(";") if c.strip()]
            out[row["query_id"]] = set(cites)
    return out

def stage_member_set(conn, family, dataset, segment_key, values):
    if not values:
        return None
    placeholders = ",".join("?" for _ in values)
    if family == "court_funnel":
        params = [dataset, segment_key, *values]
        sql = f"SELECT DISTINCT citation FROM citation_segments WHERE dataset=? AND segment_key=? AND option_value IN ({placeholders})"
    else:
        params = [dataset, family, segment_key, *values]
        sql = f"SELECT DISTINCT citation FROM citation_segments WHERE dataset=? AND family=? AND segment_key=? AND option_value IN ({placeholders})"
    return {r[0] for r in conn.execute(sql, params)}

def replay_stages(cand_path, gold_by_query):
    rows = []
    with open(cand_path, encoding="utf-8") as f:
        for line in f:
            rec = json.loads(line)
            qid = rec["query_id"]
            gold = gold_by_query.get(qid, set())
            gold_law = {c for c in gold if family_by_citation.get(c) == "law"}
            gold_court = {c for c in gold if family_by_citation.get(c) == "court"}
            for funnel, dataset, family in [("law", "laws_de", "law"), ("court", "court_considerations", "court_funnel")]:
                stages = rec["stage_diagnostics"][funnel]
                # We don't have explicit per-stage citation sets in the JSONL,
                # but final candidates are stored. Reconstruct intermediate sets
                # by intersecting source_set with each hard_filter cumulatively.
                target_gold = gold_law if funnel == "law" else gold_court
                final_set = set(rec["law_candidates"] if funnel == "law" else rec["court_candidates"])
                # Source stage = all source citations of that dataset; recall before hard filters.
                running = None
                for stage in stages:
                    name = stage["stage"]
                    if name == "source":
                        running = None  # implicit: full source
                        present_in_stage = target_gold  # all gold present in source = those with family match
                    elif name.startswith("hard_filter:"):
                        seg_key = name.split(":", 1)[1]
                        members = stage_member_set(conn, family, dataset, seg_key, stage.get("selected_option_ids") or [])
                        if members is not None:
                            running = members if running is None else running & members
                        present_in_stage = (running & target_gold) if running is not None else target_gold
                    elif name == "topk_budget":
                        present_in_stage = final_set & target_gold
                    else:
                        present_in_stage = target_gold
                    rec_recall = (len(present_in_stage) / len(target_gold)) if target_gold else None
                    rows.append({
                        "query_id": qid,
                        "funnel": funnel,
                        "stage": name,
                        "selected_options": ";".join(stage.get("selected_option_ids") or []),
                        "candidate_count": stage["candidate_count"],
                        "gold_in_target": len(target_gold),
                        "recall": rec_recall,
                    })
    return rows

gold_by_query = load_gold(RUN_SPLIT)

def trace_summary(label, cand_path):
    rows = replay_stages(cand_path, gold_by_query)
    out_path = ART_DIR / f"{RUN_SPLIT}_segment_lattice_v3_{label}_stage_recall.csv"
    with open(out_path, "w", encoding="utf-8", newline="") as f:
        w = csv.DictWriter(f, fieldnames=list(rows[0].keys()) if rows else ["query_id"])
        w.writeheader()
        w.writerows(rows)
    print(f"\n=== {label} per-stage trace → {out_path} ===")
    by_funnel_stage = {}
    for r in rows:
        by_funnel_stage.setdefault((r["funnel"], r["stage"].split(":")[0]), []).append((r["recall"], r["candidate_count"]))
    for (funnel, stage), entries in sorted(by_funnel_stage.items()):
        recalls = [e[0] for e in entries if e[0] is not None]
        counts = [e[1] for e in entries]
        if not recalls:
            continue
        print(f"  {funnel:<5} {stage:<13} | recall mean={statistics.mean(recalls):.3f} median={statistics.median(recalls):.3f} | cand median={int(statistics.median(counts)):>8}")
    return rows

if (ART_DIR / f"{RUN_SPLIT}_segment_lattice_v3_oracle_candidate_sets.jsonl").exists():
    trace_summary("oracle", ART_DIR / f"{RUN_SPLIT}_segment_lattice_v3_oracle_candidate_sets.jsonl")
if LLM_CAND.exists():
    trace_summary("llm", LLM_CAND)
conn.close()


=== oracle per-stage trace → /content/drive/MyDrive/swiss_law/artifacts/val_segment_lattice_v3_oracle_stage_recall.csv ===
  court hard_filter   | recall mean=0.440 median=0.406 | cand median=   41084
  court source        | recall mean=1.000 median=1.000 | cand median= 1985178
  court topk_budget   | recall mean=0.034 median=0.000 | cand median=    2000
  law   hard_filter   | recall mean=1.000 median=1.000 | cand median=    3757
  law   source        | recall mean=1.000 median=1.000 | cand median=  175933
  law   topk_budget   | recall mean=0.613 median=0.513 | cand median=    2000

=== llm per-stage trace → /content/drive/MyDrive/swiss_law/artifacts/val_segment_lattice_v3_llm_stage_recall.csv ===
  court hard_filter   | recall mean=0.219 median=0.000 | cand median=  174177
  court source        | recall mean=1.000 median=1.000 | cand median= 1985178
  court topk_budget   | recall mean=0.000 median=0.000 | cand median=    2000
  law   hard_filter   | recall mean=0.771 median=0.800 |

## 10. Final Summary — LLM vs Oracle

Side-by-side comparison of recall and candidate-set size. The gap between LLM and oracle is the planner deficit; closing it means improving cards or the planner prompt, not the funnel.

In [ ]:
import csv
import statistics

def load_summary(path):
    if not path.exists():
        return None
    with open(path, encoding="utf-8") as f:
        return list(csv.DictReader(f))

def aggregate(rows, label):
    if not rows:
        print(f"{label}: missing summary file"); return
    def col(name):
        return [float(r[name]) for r in rows if r.get(name) not in (None, "")]
    print(f"\n=== {label} ({len(rows)} queries) ===")
    for metric in ("recall", "law_recall", "court_recall"):
        vals = col(metric)
        if vals:
            print(f"  {metric:<13} mean={statistics.mean(vals):.3f} median={statistics.median(vals):.3f} min={min(vals):.3f}")
    for metric in ("candidate_count", "law_candidate_count", "court_candidate_count"):
        vals = [int(r[metric]) for r in rows if r.get(metric)]
        if vals:
            print(f"  {metric:<22} mean={statistics.mean(vals):.0f} median={int(statistics.median(vals))} max={max(vals)}")

aggregate(load_summary(ORACLE_SUMMARY), "ORACLE")
aggregate(load_summary(LLM_SUMMARY), "LLM PLANNER")


=== ORACLE (10 queries) ===
  recall        mean=0.396 median=0.295 min=0.191
  law_recall    mean=0.613 median=0.513 min=0.222
  court_recall  mean=0.034 median=0.000 min=0.000
  candidate_count        mean=3895 median=4000 max=4000
  law_candidate_count    mean=1895 median=2000 max=2000
  court_candidate_count  mean=2000 median=2000 max=2000

=== LLM PLANNER (10 queries) ===
  recall        mean=0.402 median=0.338 min=0.240
  law_recall    mean=0.623 median=0.562 min=0.429
  court_recall  mean=0.000 median=0.000 min=0.000
  candidate_count        mean=3814 median=4000 max=4000
  law_candidate_count    mean=1814 median=2000 max=2000
  court_candidate_count  mean=2000 median=2000 max=2000
